# Universal Data Pipeline Runner

This notebook demonstrates how to use the newly extracted modular pipeline from the `src/` directory to process either Moltbook or Reddit data.

In [1]:
import pandas as pd
import os
import sys
# Ensure src can be imported
sys.path.append(os.path.abspath('..'))

from src.data_prep import (
    get_early_comments,
    engineer_comment_existence,
    engineer_early_sentiment,
    merge_engineered_features,
    save_final_features,
    extract_hour_feature
)
from src.feature_engineering import get_vader_compound, sentiment_features, engineer_text_features
from src.embeddings import generate_and_save_embeddings, prepare_embedding_text
from src.text_processing import clean_text

c:\Users\Rald999\Anaconda3\lib\site-packages\transformers\utils\generic.py:484: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Rald999\Anaconda3\lib\site-packages\transformers\utils\generic.py:341: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Rald999\Anaconda3\lib\site-packages\transformers\utils\generic.py:341: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
# !pip install protobuf==3.20.3

## 1. Load Data

Change the paths below to point to your raw datasets.

In [ ]:
# Example Configuration (Switch parameters based on Moltbook vs Reddit)
# Uncomment and update paths!
POSTS_PATH = '../data/moltbook_cleaned_27_3.csv'
# COMMENTS_PATH = '../data/raw_comments.csv'

# Moltbook config example:
POST_ID_COL_COMMENTS = 'post_id'      
TIME_COL = 'created_utc_dt'                  
TEXT_COL_POSTS = 'content'               
TEXT_COL_COMMENTS = 'content'
POST_ID_COL_POSTS = 'id'

df_posts = pd.read_csv(POSTS_PATH)
# df_comments = pd.read_csv(COMMENTS_PATH)


## 2. Early Sentiment & Comment Engineering

In [ ]:
# 1. Extract first 10 comments
df_early = get_early_comments(df_comments, post_id_col=POST_ID_COL_COMMENTS, time_col=TIME_COL)

# 2. Calculate Comment Existence
# existence_df = engineer_comment_existence(df_early, post_id_col=POST_ID_COL_COMMENTS)

# 3. Calculate VADER sentiment over early comments
# sentiment_df = engineer_early_sentiment(
#     df_early_comments=df_early, 
#     get_vader_compound_func=get_vader_compound, 
#     post_id_col=POST_ID_COL_COMMENTS, 
#     text_col=TEXT_COL_COMMENTS
# )

# 4. Merge back into posts
# df_posts = merge_engineered_features(
#     df_posts=df_posts, 
#     existence_df=existence_df, 
#     sentiment_df=sentiment_df, 
#     post_id_col_posts=POST_ID_COL_POSTS
# )

# 5. Extract Hour Feature
df_posts = extract_hour_feature(df_posts, time_col=TIME_COL)


Extracting hour from 'created_utc_dt'...


In [ ]:
# df_posts["content"] = df_posts["title"].fillna("") + " " + df_posts["content"].fillna("")


In [6]:
df_posts = engineer_text_features(df_posts, text_col="content")


Calculating text features (vocab, stopwords, burstiness, etc.)...


In [7]:
df_posts.columns

Index(['score', 'title', 'selftext', 'forum', 'created_utc_dt',
       'comment_existence', 'avg_early_sentiment', 'max_early_sentiment',
       'min_early_sentiment', 'hour', 'content', 'ttr', 'hapax',
       'stopword_ratio', 'burstiness', 'punctuation_density', 'hedging_score',
       'self_reference_rate', 'forum_philosophy', 'forum_technology',
       'forum_todayilearned'],
      dtype='object')

In [11]:
df_posts[['score','comment_existence', 'avg_early_sentiment', 'max_early_sentiment',
       'min_early_sentiment', 'hour', 'ttr', 'hapax', 'stopword_ratio','burstiness', 'punctuation_density', 'hedging_score',
       'self_reference_rate', 'forum_philosophy', 'forum_technology',
       'forum_todayilearned']].to_csv("reddit_features_22_3_meh.csv", index=False)

In [ ]:
print)xxx

SyntaxError: unmatched ')' (2062285047.py, line 1)

In [10]:
df_posts.to_csv("reddit_22_3.csv", index=False)

## 3. Text Preparation & BERT Embeddings

In [ ]:
# Truncate and prep text
df_prep = prepare_embedding_text(df_posts, title_col='title', content_col=TEXT_COL_POSTS)

# Extract BERT Embeddings
embeddings = generate_and_save_embeddings(df_prep['safe_content'].tolist(), checkpoint_dir="../checkpoints_22_3")

# Save Final Features to Pickle (includes automatic train/val/test splits!)
final_df = save_final_features(df_prep, embeddings, output_dir="../data", dataset_prefix="processed_v1")

Combining text and enforcing token limits for BERT...


Extracting BERT Embeddings: 100%|██████████| 1090/1090 [25:34<00:00,  1.41s/it]


Final BERT Embeddings shape: (34867, 768)
[TRAIN] Saved features (shape: (24514, 773)) to ../data/processed_v1_features_train.pkl
[VAL] Saved features (shape: (5169, 773)) to ../data/processed_v1_features_val.pkl
[TEST] Saved features (shape: (5184, 773)) to ../data/processed_v1_features_test.pkl
